---
title: Lab 2, instructor solution. From estimates to a model, and back again
course: Example course (pymetrix)
links: [model-solution, least-squares, diagnostic-tests]
---

# Lab 2, instructor solution. From estimates to a model, and back again

The worked solution to the student notebook `examples/course/02_lab.ipynb`, with answers to its questions at the end. One exercise from start to finish: load data, estimate and test two equations, combine them with an identity into a model, simulate it and a policy scenario, then save everything as a project, reopen it and check that the results are the same.

The data are a simulated economy: consumption `c`, investment `i`, government spending `g` and output `y = c + i + g`.

In [ ]:
import numpy as np

from pymetrix import OLS, Dataset, Equation, Model, Scenario

data = Dataset.example("sample_macro")
data

## 1. Estimate and test

Consumption depends on income and its own past; investment on last quarter's output. Newey-West (`hac`) standard errors allow for serial correlation.

In [ ]:
cons = Equation("c ~ y + c[-1]").fit(data, method=OLS(cov="hac"))
inv = Equation("i ~ y[-1]").fit(data)
cons.summary()

In [ ]:
cons.test("breusch_godfrey", lags=4)

**Question.** Output appears on the right of the consumption equation, but output itself depends on consumption. What does that do to least squares? (Try `method=IV(...)` with `g` as an instrument.)

## 2. A model

The two estimated equations and the national-accounts identity make a model with three endogenous variables, `c`, `i` and `y`; `g` is exogenous. Solving it over 2015-2024 simulates the economy from the data before 2015.

In [ ]:
model = Model([cons, inv, Equation("y = c + i + g")])
base = model.solve(data, ("2015Q1", "2024Q4"))
error = (base["y"] - data["y"]).loc["2015Q1":"2024Q4"]
print(f"root mean squared simulation error of y: {np.sqrt((error ** 2).mean()):.2f}")

## 3. A scenario

Raise government spending by 5 from 2020 and compare. The ratio of the rise in output to the rise in spending is the multiplier.

In [ ]:
higher_g = Scenario(shift={"g": 5.0}, name="g + 5")
alt = model.solve(data, ("2020Q1", "2024Q4"), scenario=higher_g)
ref = model.solve(data, ("2020Q1", "2024Q4"))
multiplier = (alt["y"] - ref["y"]).loc["2020Q1":"2024Q4"] / 5
multiplier.round(2).head(8)

**Question.** Why is the multiplier larger after a few quarters than on impact?

## 4. Save, reopen, check

The same steps as dialog commands: each is what the menus run, and each is recorded as Python. A project saves the data and the steps; opening it replays them.

In [ ]:
from pathlib import Path
import tempfile

from pymetrix.commands import DefineEquation, FitEquation, Session, SolveModel, load_project, save_project

s = Session()
s.objects["data"] = data
s.run(FitEquation("c ~ y + c[-1]", name="cons"))
s.run(FitEquation("i ~ y[-1]", name="inv"))
s.run(DefineEquation("y = c + i + g", name="identity"))
s.run(SolveModel("cons, inv, identity", "2015Q1", "2024Q4"))
print(s.script())

In [ ]:
path = save_project(s, Path(tempfile.mkdtemp()) / "lab2.pymx")
reopened = load_project(path)
same = np.allclose(reopened["solution"]["y"], s["solution"]["y"], equal_nan=True)
print(f"saved to {path.name}; reopened results identical: {same}")

## Answers to the questions

**Simultaneity.** Output is on the right of the consumption equation, but output includes consumption, so `y` is correlated with the error and least squares is biased (upwards, for the marginal propensity to consume). Government spending moves output but not the consumption error, so it is a valid instrument: `method=IV(endog=["y"], instruments=["g"])` gives a smaller coefficient on `y`.

**A growing multiplier.** On impact, higher spending raises output, and through income, consumption, within the quarter. After that, lagged consumption and last quarter's output (in investment) carry the rise forward, so each quarter adds to the last until the effect settles near its long-run value.